# [LAB] Multi-Agent 협업 실습

Agent 여럿이 일을 나눠 맡는 그래프를 만들고 실행해 봅니다. 주제는 모두 "조사해서 글로 쓰기"입니다. 누가 조사하고, 누가 쓰고, 누가 다음 차례를 정하는지를 실행 결과로 확인하는 것이 이 노트북의 목표입니다.

- **2장 Researcher와 Writer 나누기** (과제): 조사하는 Agent와 글 쓰는 Agent를 한 그래프로 잇습니다. `# ✍️ WRITE_HERE:` 아래 빈칸 네 곳을 AI와 채웁니다.
- **3장 Supervisor가 다음 차례 정하기** (예제): Supervisor가 researcher, writer, FINISH 중 하나를 골라 흐름을 이끕니다. `# ✍️ CHANGE_HERE:` 줄을 바꿔 Supervisor의 선택이 어떻게 달라지는지 보고, AI에게 원인을 묻습니다.
- **4장 Critic 더하기** (예제, 선택): 초안의 치명적 오류를 찾는 Critic을 더합니다. 앞 장을 일찍 끝낸 분만 해 봅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모든 장이 LLM을 여러 번 호출하므로 결과가 나오기까지 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. Researcher와 Writer 나누기 (예제) (과제)
  - 2.1 검색 도구 (예제)
  - 2.2 공유 상태 (예제)
  - 2.3 Researcher 노드 (과제)
  - 2.4 도구를 쓸지 정하는 조건 (과제)
  - 2.5 Writer 노드 (예제)
  - 2.6 그래프 연결 (과제)
  - 2.7 컴파일하고 실행하기 (과제)
- 3. Supervisor가 다음 차례 정하기 (예제)
  - 3.1 라우팅 결정 형식 (예제)
  - 3.2 공유 상태 (예제)
  - 3.3 Supervisor (예제)
  - 3.4 Researcher (예제)
  - 3.5 Writer (예제)
  - 3.6 라우팅 함수 (예제)
  - 3.7 그래프 조립 (예제)
  - 3.8 실행하기 (예제)
  - 3.9 AI에게 물어보기
- 4. Critic 더하기 (예제)
  - 4.1 라우팅 결정 형식 (예제)
  - 4.2 Critic 판정 형식 (예제)
  - 4.3 공유 상태 (예제)
  - 4.4 Supervisor (예제)
  - 4.5 Researcher (예제)
  - 4.6 Writer (예제)
  - 4.7 Critic (예제)
  - 4.8 라우팅 함수 (예제)
  - 4.9 그래프 조립 (예제)
  - 4.10 stream으로 실행하기 (예제)
  - 4.11 AI에게 물어보기
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/05-multi-agent/01_researcher_writer.py`·`02_supervisor.py`·`03_supervisor_with_critic.py`. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다. 세 파일 모두 raw를 씁니다(2-2 목차 3절).
- 사외 실측(2026-09-28, `LLM_PROVIDER=openai`, `gpt-6-luna`): 01 약 40초, 02 약 20~40초, 03 약 20~30초. 사내 게이트웨이 소요 시간은 미확인입니다.
- 원본 코드 속 이전 강사 메모 주석, 주석 처리된 죽은 코드, ASCII 흐름 그림은 노트북 셀에서 뺐습니다(2026-09-28 사용자 결정). 필요한 내용은 각 장의 강사용 메모로 옮겼습니다. 원본 파일은 그대로입니다.
- 03(Critic)은 커리큘럼 2.3 항목에 없어 선택 장으로 두었습니다(2026-09-28 사용자 결정).
- 세 그래프 모두 반복 횟수를 코드로 막지 않습니다. LangGraph 기본 반복 한도가 10007이라 사실상 오류로 멈추지 않습니다. 그래서 성공 기준 ⛔에 "셀 실행 중지(■)"를 적었습니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지, 셀을 직접 고칠 수 있는지는 강의장에서 아직 확인하지 않았습니다. 못 읽으면 셀 코드를 지시 아래에 붙여 넣게 합니다.

</details>

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

"정답 보기"의 코드는 빈칸 줄과 같은 들여쓰기로 보여 줍니다. 붙여 넣을 때는 빈칸 줄 전체(줄 앞 공백까지)를 지우고 그 자리에 붙여 넣습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Researcher와 Writer 나누기 <small>(예제) (과제)</small>

조사하는 Agent(Researcher)와 글 쓰는 Agent(Writer)를 나눠 한 그래프로 잇습니다. Researcher는 검색 도구 `search_web`을 쓸 수 있고, Writer는 도구 없이 조사 결과와 지금까지의 대화를 받아 글을 씁니다. 두 Agent는 같은 상태(State)를 함께 봅니다. 조사 결과는 `research_result`, 완성된 글은 `draft`에 담깁니다.

검색 도구는 실제로 검색하지 않고, 정해 둔 글(검색 결과 흉내)을 돌려줍니다.

빈칸은 네 곳입니다. 그래프는 맨 아래 "컴파일하고 실행하기" 셀에서 처음 실행되므로, **네 빈칸을 모두 채운 뒤에** 결과를 볼 수 있습니다. 빈칸을 채운 셀은 그때마다 실행해 두고, 마지막 셀에서 한꺼번에 확인합니다.

**왜 배우나:** 역할을 나누면 Agent마다 필요한 도구와 지시만 갖게 됩니다. Researcher만 도구를 쓰고 Writer는 쓰지 않는 구조를 직접 만들어 보면, 역할 분리가 코드에서 어떤 모양인지 알 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/05-multi-agent/01_researcher_writer.py`. 솔루션은 researcher 시스템 프롬프트가 `'너는 전문 리서처인데, 검색이 필요하면 . '`로 끊겨 있어 raw를 썼습니다. raw는 조사 결과(`research_result`)도 출력해 분업이 화면에 보입니다.
- 셀에서 뺀 원본 메모의 요지:
  - State: 메시지는 목록에 차례로 쌓이므로, 도구 결과를 나중에 다시 쓰려면 찾아와야 한다. 그래서 `research_result`에 따로 담는다.
  - researcher_node: `[system] + state['messages']`는 리스트 더하기다(`[1, 2, 3] + [4, 5, 6] = [1, 2, 3, 4, 5, 6]`).
  - researcher_condition: 마지막 메시지(대개 AIMessage)에 `tool_calls`가 있으면 도구를 호출하라는 요청이 가장 최근이라는 뜻이다.
  - writer_node: Researcher와 같은 모델이지만 도구를 연결하지 않았으므로 `tool_calls`가 생기지 않는다.
  - 엣지: LangGraph의 `tools_condition`은 tools 또는 END로만 가르므로, writer로 보내려고 직접 만든 조건 함수를 쓴다.
  - 주석 처리된 `app.stream(...)` 블록과 끝의 ASCII 흐름 그림(START → researcher ⇄ tools → writer, state는 그래프 안에서 공유)은 뺐습니다.
- 솔루션 끝 문자열의 토론 메모 요지: 한 Agent에 웹검색과 답변 작성 두 일을 맡기면 통과 기준을 프롬프트 하나에 다 넣어야 한다. 작성자를 떼어 따로 관리하면 역할마다 기준이 단순해지지만, 두 Agent를 잇는 고리(State)를 설계해야 한다. 슬라이드 "역할 분리와 책임 경계"와 같은 이야기입니다.
- 사외 실측에서 Researcher가 검색어를 영어로 만든 적이 있습니다. 검색 결과 첫 줄의 따옴표 안 글은 실행마다 다릅니다.

</details>

### 2.1 검색 도구 <small>(예제)</small>

In [ ]:
# ── 도구 ──────────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 조사한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f'["{query}" 검색 결과]\n'
        f"- 해당 주제의 주요 동향: 최근 급격한 성장세를 보이고 있음\n"
        f"- 핵심 기술 요소: 트랜스포머 아키텍처, 대규모 데이터 학습\n"
        f"- 주요 플레이어: OpenAI, Anthropic, Google DeepMind\n"
        f"- 미래 전망: 향후 3년간 지속적인 발전이 예상됨"
    )

### 2.2 공유 상태 <small>(예제)</small>

In [ ]:
# ── 상태 ──────────────────────────────────────────────────────────────────────
class State(TypedDict):
    messages: Annotated[list, add_messages]
    research_result: str  # Researcher가 수집한 정보
    draft: str  # Writer가 작성한 초안

### 2.3 Researcher 노드 <small>(과제)</small>

**✍️ WRITE_HERE: Researcher LLM을 호출하고 응답 메시지를 반환해 주세요.**

`researcher_node`는 Researcher가 한 번 생각하는 단계입니다. 시스템 메시지(역할)와 지금까지 쌓인 메시지를 도구가 연결된 모델 `researcher_llm`에 보내고, 받은 응답을 상태에 더해야 합니다. 응답에 도구 호출 요청이 들어 있으면 다음 단계에서 검색 도구가 실행됩니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

**프롬프트 힌트** 이 셀의 `researcher_node` 함수를 AI 채팅창에 붙여 넣고, 시스템 메시지와 상태의 메시지 목록을 함께 `researcher_llm`에 보내 달라고 합니다. 받은 응답은 상태의 `messages`에 더해지는 딕셔너리로 돌려 달라고 합니다. 함수 밖의 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 이 셀이 오류 없이 끝나고 아무것도 출력하지 않는다
- 🟢 마지막 셀을 실행했을 때 "=== 최종 글 ===" 아래에 `["…" 검색 결과]`로 시작하는 줄이 나온다
- ⛔ 마지막 셀에서 오류가 나고 오류 글에 `researcher`가 보인다 → 함수가 딕셔너리를 돌려주는지 다시 봅니다

In [ ]:
# ── Researcher Agent ──────────────────────────────────────────────────────────
# 역할: 주제를 조사해 research_result에 저장한다.

researcher_llm = llm.bind_tools([search_web])
researcher_tool_node = ToolNode([search_web])


def researcher_node(state: State) -> dict:
    # LLM의 역할 (페르소나) 정의
    system = SystemMessage(
        content=(
            "너는 전문 리서처야. "
            "search_web 도구를 사용해 주어진 주제를 철저히 조사하고, "
            "수집한 정보를 정리해서 제공해줘."
        )
    )
    # ✍️ WRITE_HERE: Researcher LLM을 호출하고 응답 메시지를 반환해 주세요.
    ...  # ✍️ 여기를 채웁니다


def researcher_tools_node(state: State) -> dict:
    # ToolNode 실행 후 research_result에 도구 결과를 저장
    tool_result = researcher_tool_node.invoke(state)
    last_tool_msg = tool_result["messages"][-1]
    return {
        "messages": tool_result["messages"],
        "research_result": last_tool_msg.content,
    }

<details><summary>정답 보기</summary>

```python
    response = researcher_llm.invoke([system] + state["messages"])
    return {"messages": [response]}
```

</details>

### 2.4 도구를 쓸지 정하는 조건 <small>(과제)</small>

**✍️ WRITE_HERE: 마지막 메시지의 tool_calls 여부에 따라 researcher_tools 또는 writer를 반환해 주세요.**

Researcher가 응답한 뒤 어디로 갈지 정하는 조건 함수입니다. 마지막 메시지에 도구 호출 요청(`tool_calls`)이 있으면 검색 도구 노드 `researcher_tools`로, 없으면 조사가 끝난 것이므로 `writer`로 보냅니다. 돌려주는 문자열은 그래프에 등록할 노드 이름과 똑같아야 합니다.

**프롬프트 힌트** `researcher_condition` 함수를 AI 채팅창에 붙여 넣고, 마지막 메시지에 도구 호출 요청이 있는지에 따라 다음 노드 이름을 돌려 달라고 합니다. 도구 호출 요청이 있으면 `researcher_tools`, 없으면 `writer`를 돌려 달라고 적습니다.

**성공 기준**

- 🟢 마지막 셀에서 검색 결과 줄들 다음에 서론·본론·결론으로 된 글이 이어진다
- ⛔ 검색 결과 줄 없이 글만 나온다 → 도구를 거치지 않고 writer로 갔습니다. 조건을 다시 봅니다
- ⛔ 마지막 셀이 몇 분이 지나도 끝나지 않는다 → 셀 실행을 중지(■)하고 조건이 writer를 돌려줄 수 있는지 봅니다

In [ ]:
def researcher_condition(state: State):
    """Researcher가 도구 호출을 완료하면 writer로 넘어간다."""
    last = state["messages"][-1]

    # ✍️ WRITE_HERE: 마지막 메시지의 tool_calls 여부에 따라 researcher_tools 또는 writer를 반환해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "researcher_tools"
    return "writer"
```

</details>

### 2.5 Writer 노드 <small>(예제)</small>

In [ ]:
# ── Writer Agent ──────────────────────────────────────────────────────────────
# 역할: Researcher의 조사 결과를 바탕으로 완성된 글을 작성한다.


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 작가야. "
            "제공된 리서치 결과를 바탕으로 독자가 이해하기 쉬운 글을 작성해줘. "
            "서론, 본론, 결론 구조를 갖춰줘."
        )
    )
    context = f"[리서치 결과]\n{state.get('research_result', '')}"
    response = llm.invoke(
        [
            system,
            HumanMessage(content=context),
            *state["messages"],
        ]
    )
    return {"messages": [response], "draft": response.content}

### 2.6 그래프 연결 <small>(과제)</small>

**✍️ WRITE_HERE: START, 조건부 엣지, researcher_tools 반복, writer 종료 엣지를 연결해 주세요.**

노드 세 개(`researcher`, `researcher_tools`, `writer`)는 등록되어 있습니다. 이제 흐름을 잇습니다. 시작하면 researcher로 가고, researcher 다음은 위에서 만든 조건 함수가 정하고, 검색 도구를 실행한 뒤에는 다시 researcher로 돌아오고, writer가 끝나면 그래프도 끝납니다.

**프롬프트 힌트** 이 셀 전체를 AI 채팅창에 붙여 넣고, 위 설명의 네 가지 흐름을 엣지로 연결해 달라고 합니다. researcher 다음 갈림길에는 `researcher_condition` 함수를 쓰고, 노드 이름은 바꾸지 말라고 적습니다.

**성공 기준**

- 🟢 이 셀이 오류 없이 끝난다(셀 아래에 `<langgraph.graph.state.StateGraph at 0x…>` 한 줄이 보여도 정상이다)
- 🟢 마지막 셀에서 검색 결과 줄과 글이 모두 나온다
- ⛔ 마지막 셀의 컴파일 줄에서 오류가 난다 → 시작(START) 엣지가 빠졌거나 노드 이름이 다릅니다

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("researcher", researcher_node)
builder.add_node("researcher_tools", researcher_tools_node)
builder.add_node("writer", writer_node)

# ✍️ WRITE_HERE: START, 조건부 엣지, researcher_tools 반복, writer 종료 엣지를 연결해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", researcher_condition)
builder.add_edge("researcher_tools", "researcher")
builder.add_edge("writer", END)
```

</details>

### 2.7 컴파일하고 실행하기 <small>(과제)</small>

**✍️ WRITE_HERE: 그래프를 컴파일해 주세요.**

연결을 마친 그래프를 실행할 수 있는 앱으로 만듭니다. 그 아래 코드는 요청 한 문장을 넣어 그래프를 실행하고, 조사 결과와 완성된 글을 차례로 출력합니다. 앞의 세 빈칸(2.3, 2.4, 2.6)을 모두 채운 뒤 실행합니다.

**프롬프트 힌트** 빈칸 줄만 AI 채팅창에 붙여 넣고, 연결을 마친 `builder`를 실행할 수 있는 그래프로 만들어 `app`에 담아 달라고 합니다. 아래의 실행 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 "=== 최종 글 ===" 아래에 가짜 검색 결과 다섯 줄(`주요 플레이어: OpenAI, Anthropic, Google DeepMind` 줄 포함)이 먼저 나온다
- 🟢 그 아래에 서론·본론·결론으로 된 글이 나온다
- ⛔ `app`이 정의되지 않았다는 오류가 난다 → 빈칸이 `app`에 담는지 봅니다
- ⛔ 셀이 몇 분이 지나도 끝나지 않는다 → 셀 실행을 중지(■)하고 조건 함수와 엣지를 다시 봅니다

In [ ]:
# ✍️ WRITE_HERE: 그래프를 컴파일해 주세요.
...  # ✍️ 여기를 채웁니다

result = app.invoke(
    {
        "messages": [
            HumanMessage(content="AI Agent 기술의 현황과 전망을 조사해서 글로 써줘.")
        ],
        "research_result": "",
        "draft": "",
    }
)

print("=== 최종 글 ===")
print(result["research_result"])
print(result["draft"])

<details><summary>정답 보기</summary>

```python
app = builder.compile()
```

</details>

**볼 지점**
- "=== 최종 글 ===" 바로 아래 다섯 줄은 Researcher가 검색 도구로 받아 `research_result`에 담은 조사 결과입니다. 첫 줄 따옴표 안이 Researcher가 만든 검색어입니다.
- 그 아래 글은 Writer가 조사 결과와 지금까지의 대화를 받아 쓴 `draft`입니다. 조사 결과의 "주요 플레이어" 이름이 글에 없을 수도 있습니다. Writer는 조사 결과를 바탕으로 하되, 모델이 아는 내용도 섞어 씁니다.
- 흐름은 researcher → researcher_tools → researcher → writer 순서입니다. Researcher가 도구 결과를 보고 한 번 더 생각한 뒤 조건 함수가 writer로 보냅니다.

## 3. Supervisor가 다음 차례 정하기 <small>(예제)</small>

이번에는 Supervisor가 다음에 누가 일할지 정합니다. Supervisor는 매번 researcher, writer, FINISH 중 하나를 고르고, 고른 이유를 함께 남깁니다. 하위 Agent는 일을 마치면 다시 Supervisor에게 돌아옵니다. 이 실습의 Supervisor는 다음 차례만 고르고 결과를 종합하지는 않습니다. 최종 글은 Writer가 쓴 초안입니다.

Supervisor의 선택은 `RouteDecision`이라는 정해진 형식으로 받습니다. `next`에는 세 값 중 하나만 들어갈 수 있어서, 모델이 엉뚱한 이름을 만들어 내도 그래프가 길을 잃지 않습니다. 이번 장의 Researcher는 도구 없이 모델이 알고 있는 내용으로 조사합니다.

코드는 완성되어 있습니다. 먼저 그대로 끝까지 실행하고, 그 출력으로 3.9 "AI에게 물어보기"를 먼저 합니다. 그다음 돌아와 `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행해 결과를 비교합니다. 다시 실행하면 처음 출력이 지워지기 때문입니다.

**왜 배우나:** 앞 장은 다음 차례를 조건 함수(코드)가 정했습니다. 이번 장은 모델이 정합니다. 모델에게 맡긴 결정이 어떻게 흔들릴 수 있는지 보고, 그 원인을 코드에서 찾는 연습을 합니다.

이 장의 준비 코드입니다. 앞 장과 같은 모델을 쓰고, 정해진 형식으로 답을 받는 데 필요한 `BaseModel`을 더 불러옵니다. 실행만 합니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/05-multi-agent/02_supervisor.py`. 솔루션에는 쓰지 않는 import(`tool`, `ToolNode`)가 더 있어 raw를 썼습니다.
- 셀에서 뺀 원본 메모의 요지:
  - 라우팅 스키마: 어느 노드로 갈지를 LLM이 자유 문장으로 정하지 않고, `next` 속성의 값(노드 이름 문자열)으로 정하게 한다.
  - SUPERVISOR_PROMPT: 프롬프트만으로 제어하는 데는 한계가 있어 Pydantic으로 출력 형식을 고정했다.
  - supervisor_node: `reason`에는 "조사가 부족하다" 같은 판단 이유가 들어간다.
  - researcher_node: 앞 파일은 마지막 메시지를 넣었지만 여기서는 첫 메시지(사용자 요청)를 주제로 쓴다. 원본 메모도 "올바른 방법은 아니다"라고 적었다.
  - `add_conditional_edges('supervisor', route)`: 끝 노드가 라우팅 함수의 결과로 정해지는 add_edge다.
- 사외 실측(`gpt-6-luna`): 원본 그대로 4번 중 3번은 Supervisor가 researcher 다음 곧바로 FINISH를 골라 "최종 초안"이 비었습니다. Researcher가 조사 결과 대신 완성된 블로그 글을 써서 메시지에 쌓았고, Supervisor가 이를 초안으로 본 것으로 보입니다. CHANGE_HERE 줄에 "핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마."를 더하자 2번 모두 researcher → writer → FINISH로 갔습니다. 사내 모델에서의 빈도는 미확인입니다. 그래서 CHANGE_HERE를 목차의 사용자 요청 줄 대신 Researcher 지시 줄로 옮겼습니다.
- 사내 게이트웨이에서 `with_structured_output`(json_schema 방식) 동작은 미확인입니다. 실패하면 이 장은 코드 설명과 AI에게 물어보기로 진행합니다.

</details>

### 3.1 라우팅 결정 형식 <small>(예제)</small>

In [ ]:
# ── Supervisor가 내리는 라우팅 결정 스키마 ─────────────────────────────────────
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]
    reason: str


# 수퍼바이저를 위한 LLM을 하나 정의
supervisor_llm = llm.with_structured_output(RouteDecision)

### 3.2 공유 상태 <small>(예제)</small>

In [ ]:
# ── 공유 State ────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str  # Supervisor가 결정한 다음 에이전트
    research_result: str
    draft: str

### 3.3 Supervisor <small>(예제)</small>

In [ ]:
# ── Supervisor ────────────────────────────────────────────────────────────────
# 사용자의 요청과 현재 진행 상황을 보고 어떤 에이전트를 호출할지 결정한다.

SUPERVISOR_PROMPT = """너는 멀티 에이전트 시스템의 Supervisor야.
다음 에이전트 중 하나를 선택해 작업을 지시하거나 완료를 선언해줘.

- researcher: 웹 검색으로 정보를 수집해야 할 때
- writer: 수집된 정보를 바탕으로 글을 작성해야 할 때
- FINISH: 모든 작업이 완료됐을 때

현재 상황:
- 리서치 완료 여부: {research_done}
- 초안 작성 여부: {draft_done}
"""


def supervisor_node(state: State) -> dict:
    prompt = SUPERVISOR_PROMPT.format(
        research_done="완료" if state.get("research_result") else "미완료",
        draft_done="완료" if state.get("draft") else "미완료",
    )
    decision = supervisor_llm.invoke(
        [
            SystemMessage(content=prompt),
            *state["messages"],
        ]
    )
    return {
        "next": decision.next,
        "messages": [
            AIMessage(content=f"[Supervisor → {decision.next}] {decision.reason}")
        ],
    }

### 3.4 Researcher <small>(예제)</small>

**✍️ CHANGE_HERE: Researcher가 돌려줄 결과의 형식을 정해 봅니다**

먼저 이 셀을 그대로 두고 맨 아래 "실행하기"까지 실행해 결과를 봅니다. 그 출력으로 3.9 "AI에게 물어보기"를 먼저 하고 돌아옵니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 괄호 안 지시 끝에 Researcher가 돌려줄 형식을 더합니다(예: "핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마."). 따옴표와 괄호는 지우지 않습니다. 바꾼 뒤에는 **이 셀부터 아래 셀을 차례로 다시 실행**해야 그래프에 반영됩니다.

In [ ]:
# ── Researcher ────────────────────────────────────────────────────────────────


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 리서처야. 주제를 조사해서 핵심 정보를 정리해줘."
    )
    topic = state["messages"][0].content  # 사용자의 원래 요청
    result = llm.invoke(
        [
            system,
            HumanMessage(
                # ✍️ CHANGE_HERE: Researcher가 돌려줄 결과의 형식을 정해 봅니다
                content=f"다음 주제를 조사해줘: {topic}\n(실제 검색 대신 알고 있는 내용으로 답해줘.)"
            ),
        ]
    )
    return {
        "research_result": result.content,
        "messages": [AIMessage(content=f"[Researcher 완료]\n{result.content}")],
    }

### 3.5 Writer <small>(예제)</small>

In [ ]:
# ── Writer ────────────────────────────────────────────────────────────────────


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 작가야. 리서치 결과를 바탕으로 완성도 높은 글을 써줘."
    )
    result = llm.invoke(
        [
            system,
            HumanMessage(
                content=f"다음 리서치 결과로 글을 써줘:\n\n{state['research_result']}"
            ),
        ]
    )
    return {
        "draft": result.content,
        "messages": [AIMessage(content=f"[Writer 완료]\n{result.content}")],
    }

### 3.6 라우팅 함수 <small>(예제)</small>

In [ ]:
# ── 라우팅 함수 ───────────────────────────────────────────────────────────────
def route(state: State) -> Literal["researcher", "writer", "__end__"]:
    return state["next"] if state["next"] != "FINISH" else END

### 3.7 그래프 조립 <small>(예제)</small>

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route)
builder.add_edge("researcher", "supervisor")  # 완료 후 Supervisor에게 보고
builder.add_edge("writer", "supervisor")  # 완료 후 Supervisor에게 보고

app = builder.compile()

### 3.8 실행하기 <small>(예제)</small>

In [ ]:
result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="LangGraph Multi-Agent 시스템에 대한 블로그 포스트를 써줘."
            )
        ],
        "next": "",
        "research_result": "",
        "draft": "",
    }
)

print("=== Supervisor 의사결정 흐름 ===")
for msg in result["messages"]:
    if (
        "[Supervisor" in msg.content
        or "[Researcher" in msg.content
        or "[Writer" in msg.content
    ):
        print(
            f"  {msg.content[:100]}..."
            if len(msg.content) > 100
            else f"  {msg.content}"
        )

print("\n=== 최종 초안 ===")
print(result["draft"])

**성공 기준**

- 🟢 처음 실행: `→ researcher`, `[Researcher 완료]` 다음이 `→ writer` 없이 `→ FINISH`여도("최종 초안" 아래가 빔), `→ writer`여도 정상이다
- 🟢 CHANGE_HERE 줄을 바꾼 뒤: `→ researcher`, `[Researcher 완료]`, `→ writer`, `[Writer 완료]`, `→ FINISH`가 이 순서로 나오고, "최종 초안" 아래에 글이 출력된다

**볼 지점**
- "Supervisor 의사결정 흐름"의 `[Supervisor → 이름]` 줄이 Supervisor가 고른 다음 차례와 이유입니다. 화살표 뒤 이름은 researcher, writer, FINISH 셋 중 하나만 나옵니다.
- 처음 실행에서는 `→ writer` 없이 `→ FINISH`로 끝나고 "최종 초안" 아래가 비어 있는 경우가 대부분입니다. 이때 `[Researcher 완료]` 줄이 조사 결과가 아니라 완성된 글처럼 시작하는지 봅니다. 원인은 아래 "AI에게 물어보기"에서 찾습니다.
- CHANGE_HERE 줄을 바꾼 뒤에는 Researcher 결과가 짧은 목록이 되고, Supervisor가 writer를 고르는지 비교합니다. 결과는 모델이 정하므로 실행할 때마다 다를 수 있습니다.
- ⛔ 셀이 몇 분이 지나도 끝나지 않으면 같은 Agent를 되풀이하고 있는 것입니다. 셀 실행을 중지(■)하고 CHANGE_HERE 줄을 다시 봅니다.

### 3.9 AI에게 물어보기

**프롬프트 힌트** Supervisor 셀, 라우팅 함수 셀, 처음 실행한 "의사결정 흐름" 출력을 붙여 넣고, 초안 작성이 "미완료"인데도 FINISH가 나올 수 있는 이유를 코드에서 찾아 달라고 합니다. 코드는 고치지 말고 근거가 되는 줄을 짚어 달라고 적습니다.

**성공 기준**

- 🟢 다음 차례는 코드가 아니라 Supervisor 모델이 고른 `next` 값으로 정해지고, 라우팅 함수는 그 값을 따르기만 한다는 점을 짚었다
- 🟢 Researcher의 결과가 메시지에 쌓여 Supervisor가 그것을 완성된 글로 볼 수 있다는 점을 짚었다
- 🟢 FINISH 전에 writer를 반드시 거치게 하는 코드가 없다는 점을 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다
- ⛔ "모델이 가끔 실수한다"로만 끝냈다 → 어느 줄의 어떤 값이 Supervisor에게 전달되는지 다시 묻습니다

## 4. Critic 더하기 <small>(예제)</small>

이 장은 선택입니다. 앞 장을 일찍 끝낸 분만 해 봅니다.

앞 장의 그래프에 Critic을 더합니다. Critic은 결과물을 검토하는 Reviewer 역할의 Agent입니다. Critic은 초안에서 사실 오류, 논리 모순, 핵심 요구 누락 같은 **치명적 오류만** 찾습니다. 오류를 찾으면 Supervisor가 writer에게 수정을 맡기고, 수정한 초안을 다시 Critic이 봅니다. Writer는 처음 쓸 때 일부러 치명적인 오류를 섞으라는 지시를 받습니다. Critic이 오류를 잡아내는지 보려는 장치입니다.

실행은 `stream`으로 합니다. 노드가 하나 끝날 때마다 `[노드 이름]`과 결과의 앞 100자를 출력합니다.

코드는 완성되어 있습니다. 먼저 그대로 실행하고, `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행해 흐름을 비교합니다.

이 장의 준비 코드도 앞 장과 거의 같습니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/05-multi-agent/03_supervisor_with_critic.py`. raw와 솔루션이 같습니다. 쓰지 않는 import(`tool`, `ToolNode`)가 접힌 준비 코드에 남아 있습니다.
- Critic은 슬라이드 "역할 분리와 책임 경계"의 Reviewer(누락·오류·근거 일치로 평가)에 해당합니다.
- 사외 실측(`gpt-6-luna`): 원본 그대로 3번 모두 Supervisor가 researcher 다음 writer를 건너뛰고 critic을 골랐고, Critic은 "초안이 제공되지 않아 분석할 수 없다"고 답했습니다(`draft`가 빈 문자열). Researcher 지시 줄을 바꾸자 2번 모두 researcher → writer → critic 순서가 나왔고, 1번은 Critic이 논리 모순을 찾아 writer가 한 번 고친 뒤 끝났습니다. 사내 모델은 미확인입니다.
- `revision_count` 주석은 "무한 루프 방지"지만, 실제로 멈추는 조건은 SUPERVISOR_PROMPT의 "수정 횟수가 4회를 넘었다면 FINISH" 문장뿐이고 `route`에는 없습니다. 이 점을 AI에게 물어보기의 리뷰 소재로 씁니다(2026-09-28 사용자 결정).

</details>

### 4.1 라우팅 결정 형식 <small>(예제)</small>

In [ ]:
# ── Supervisor 가 내리는 라우팅 결정 스키마 ─────────────────────────────────────


class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "critic", "FINISH"]
    reason: str


supervisor_llm = llm.with_structured_output(RouteDecision)

### 4.2 Critic 판정 형식 <small>(예제)</small>

In [ ]:
# ── Critic 이 분석하는 치명적 오류 스키마 ──────────────────────────────────────


class CriticFeedback(BaseModel):
    has_fatal_error: bool
    error_type: str  # 'factual_error', 'logical_error', 'missing_key_point', 'none'
    description: str
    suggestion: str


critic_llm = llm.with_structured_output(CriticFeedback)

### 4.3 공유 상태 <small>(예제)</small>

In [ ]:
# ── 공유 State ────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str  # Supervisor 가 결정한 다음 에이전트
    research_result: str
    draft: str
    critic_feedback: str  # Critic 의 개선점 피드백
    revision_count: int  # 수정 횟수 (무한 루프 방지)

### 4.4 Supervisor <small>(예제)</small>

In [ ]:
# ── Supervisor ────────────────────────────────────────────────────────────────
# 사용자의 요청과 현재 진행 상황을 보고 어떤 에이전트를 호출할지 결정한다.

SUPERVISOR_PROMPT = """너는 멀티 에이전트 시스템의 Supervisor 야.
다음 에이전트 중 하나를 선택해 작업을 지시하거나 완료를 선언해줘.

- researcher: 웹 검색으로 정보를 수집해야 할 때
- writer: 수집된 정보를 바탕으로 글을 작성해야 할 때
- critic: 작성된 초안의 치명적 오류를 분석해야 할 때
- FINISH: 모든 작업이 완료됐을 때

현재 상황:
- 리서치 완료 여부: {research_done}
- 초안 작성 여부: {draft_done}
- Critic 피드백 존재: {critic_done}
- 현재 수정 횟수: {revision_count}

주의:
- Critic 이 치명적 오류를 찾았다면 writer 를 호출해서 수정하게 해줘.
- 수정 횟수가 4 회를 넘었다면 FINISH 를 선택해줘.
"""


def supervisor_node(state: State) -> dict:
    prompt = SUPERVISOR_PROMPT.format(
        research_done="완료" if state.get("research_result") else "미완료",
        draft_done="완료" if state.get("draft") else "미완료",
        critic_done="있음" if state.get("critic_feedback") else "없음",
        revision_count=state.get("revision_count", 0),
    )
    decision = supervisor_llm.invoke(
        [
            SystemMessage(content=prompt),
            *state["messages"],
        ]
    )
    return {
        "next": decision.next,
        "messages": [
            AIMessage(content=f"[Supervisor → {decision.next}] {decision.reason}")
        ],
    }

### 4.5 Researcher <small>(예제)</small>

**✍️ CHANGE_HERE: Researcher가 돌려줄 결과의 형식을 정해 봅니다**

먼저 그대로 맨 아래 "stream으로 실행하기"까지 실행해 흐름을 봅니다. 그다음 앞 장에서처럼 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 괄호 안 지시 끝에 Researcher가 돌려줄 형식을 더합니다. 이 셀부터 아래 셀을 차례로 다시 실행합니다.

In [ ]:
# ── Researcher ────────────────────────────────────────────────────────────────


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 리서처야. 주제를 조사해서 핵심 정보를 정리해줘."
    )
    topic = state["messages"][0].content  # 사용자의 원래 요청
    result = llm.invoke(
        [
            system,
            HumanMessage(
                # ✍️ CHANGE_HERE: Researcher가 돌려줄 결과의 형식을 정해 봅니다
                content=f"다음 주제를 조사해줘: {topic}\n(실제 검색 대신 알고 있는 내용으로 답해줘.)"
            ),
        ]
    )
    return {
        "research_result": result.content,
        "messages": [AIMessage(content=f"[Researcher 완료]\n{result.content}")],
    }

### 4.6 Writer <small>(예제)</small>

**✍️ CHANGE_HERE: Writer에게 준 "일부러 오류 섞기" 지시를 빼 봅니다**

Researcher 줄을 바꾼 뒤 흐름에 `[writer]`가 나오면, 이번에는 `# ✍️ CHANGE_HERE:` 바로 아래 줄에서 "단 의도적으로 치명적인 오류를 섞어 넣어줘"를 지웁니다. 따옴표는 남깁니다. 이 셀부터 아래 셀을 다시 실행하고, Critic의 판정이 달라지는지 봅니다.

In [ ]:
# ── Writer ────────────────────────────────────────────────────────────────────


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 작가야. 리서치 결과를 바탕으로 완성도 높은 글을 써줘."
    )

    # Critic 피드백이 있다면 이를 반영하여 수정
    if state.get("critic_feedback"):
        prompt = f"""
        다음 리서치 결과로 글을 수정해줘.

        [기존 초안]
        {state["draft"]}

        [Critic 피드백 - 치명적 오류]
        {state["critic_feedback"]}

        위 피드백을 반영하여 치명적 오류를 수정한 새로운 버전을 작성해줘.
        """
    else:
        # ✍️ CHANGE_HERE: Writer에게 준 "일부러 오류 섞기" 지시를 빼 봅니다
        prompt = f"다음 리서치 결과로 글을 써줘:\n\n{state['research_result']}. 단 의도적으로 치명적인 오류를 섞어 넣어줘"

    result = llm.invoke(
        [
            system,
            HumanMessage(content=prompt),
        ]
    )
    return {
        "draft": result.content,
        "critic_feedback": "",
        "revision_count": state.get("revision_count", 0) + 1,
        "messages": [
            AIMessage(
                content=f"[Writer 완료 - 수정 {state.get('revision_count', 0) + 1}번째]\n{result.content}"
            )
        ],
    }

### 4.7 Critic <small>(예제)</small>

In [ ]:
# ── Critic ────────────────────────────────────────────────────────────────────

CRITIC_PROMPT = """너는 치명적 오류만 분석하는 Critic 이야.
다음 기준에서 **치명적 오류**만 찾아내:
1. 사실적 오류 (Fact-checking 실패)
2. 논리적 모순 (앞뒤 내용이 맞지 않음)
3. 핵심 요구사항 누락 (사용자 요청을 충족하지 못함)

사소한 문체, 오타, 스타일 문제는 무시해.
치명적 오류가 없다면 has_fatal_error=False 로 응답해.
"""


def critic_node(state: State) -> dict:
    feedback = critic_llm.invoke(
        [
            SystemMessage(content=CRITIC_PROMPT),
            HumanMessage(content=f"다음 초안을 분석해줘:\n\n{state['draft']}"),
        ]
    )

    if feedback.has_fatal_error:
        critic_msg = f"[치명적 오류 발견: {feedback.error_type}]\n{feedback.description}\n\n[개선 제안]\n{feedback.suggestion}"
    else:
        critic_msg = "[치명적 오류 없음]\n초안이 양호합니다."

    return {
        "critic_feedback": critic_msg if feedback.has_fatal_error else "",
        "messages": [AIMessage(content=f"[Critic 완료]\n{critic_msg}")],
    }

### 4.8 라우팅 함수 <small>(예제)</small>

In [ ]:
# ── 라우팅 함수 ───────────────────────────────────────────────────────────────


def route(state: State) -> Literal["researcher", "writer", "critic", "__end__"]:
    return state["next"] if state["next"] != "FINISH" else END

### 4.9 그래프 조립 <small>(예제)</small>

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("critic", critic_node)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route)

# 각 노드 완료 후 Supervisor 에게 보고
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")
builder.add_edge("critic", "supervisor")
app = builder.compile()

### 4.10 stream으로 실행하기 <small>(예제)</small>

In [ ]:
# ── stream 형식으로 실행 (각 노드별 실시간 진행 확인) ─────────────────────────

print("=== Agent 실행 시작 ===")

for chunk in app.stream(
    {
        "messages": [
            HumanMessage(
                content="LangGraph Multi-Agent 시스템에 대한 블로그 포스트를 써줘."
            )
        ],
        "next": "",
        "research_result": "",
        "draft": "",
        "critic_feedback": "",
        "revision_count": 0,
    }
):
    for node_name in chunk.keys():
        print(f"========= [{node_name}] =========")
        message_type = chunk[node_name]["messages"][0].__class__.__name__
        if message_type == "AIMessage":
            print("AIMessage")
        elif message_type == "ToolMessage":
            print("ToolMessage")
        print(chunk[node_name]["messages"][0].content[:100])

**성공 기준**

- 🟢 처음 실행: `[researcher]` 다음에 `[writer]` 없이 `[critic]`이 나오고, Critic이 초안이 없다고 답해도 정상이다
- 🟢 Researcher 줄을 바꾼 뒤 `[researcher]` 다음에 `[writer]`가 나오고("수정 1번째"), 그다음 `[critic]`이 나온다
- 🟢 `[critic]` 블록에 "치명적 오류 발견"이 나오면 뒤이어 `[writer]`("수정 2번째")가 나오고, "치명적 오류 없음"이 나오면 곧 끝난다
- 🟢 Writer 지시에서 오류 섞기를 뺀 뒤에는 "치명적 오류 없음"이 더 쉽게 나온다

**볼 지점**
- `=========` 사이의 이름이 방금 끝난 노드입니다. supervisor가 다른 노드 사이사이에 끼어 다음 차례를 정합니다.
- 원본 그대로 실행하면 `[writer]` 없이 `[critic]`이 먼저 나오고 "초안이 제공되지 않아" 같은 말이 보일 수 있습니다. 앞 장과 같은 원인입니다.
- `[writer]` 블록의 "수정 N번째"가 몇까지 올라가는지 봅니다.
- ⛔ writer와 critic이 다섯 번 넘게 오가며 끝나지 않으면 셀 실행을 중지(■)합니다. 왜 멈추지 않는지는 아래 "AI에게 물어보기"에서 찾습니다.

### 4.11 AI에게 물어보기

**프롬프트 힌트** Supervisor 셀, Writer 셀, 라우팅 함수 셀을 붙여 넣고, writer와 critic의 수정 반복이 반드시 끝난다는 보장이 코드 어디에 있는지 리뷰해 달라고 합니다. 코드는 고치지 말고, 보장이 코드에 있는지 프롬프트 문장에만 있는지 나눠 답해 달라고 적습니다.

**성공 기준**

- 🟢 "4회를 넘으면 FINISH"는 Supervisor 프롬프트의 문장일 뿐이고 `route` 함수에는 횟수 검사가 없다는 점을 짚었다
- 🟢 `revision_count`는 Writer가 올리기만 하고, 그 값으로 흐름을 막는 코드는 없다는 점을 짚었다
- ⛔ "`revision_count`가 있으니 무한 반복은 없다"고 답했다 → 그 값을 검사하는 줄이 어디 있는지 다시 묻습니다
- ⛔ 코드를 고쳐 왔다 → 고치지 말라고 다시 적습니다

## 5. 정리

Agent를 나누고, 다음 차례를 정하는 방법을 세 가지로 보았습니다.

| 장 | 다음 차례를 정하는 것 | 확인한 것 |
|---|---|---|
| 2. Researcher와 Writer 나누기 | 조건 함수(코드)가 `tool_calls`를 보고 정함 | Researcher만 도구를 쓰고, 조사 결과를 Writer가 받아 글을 씀 |
| 3. Supervisor가 다음 차례 정하기 | Supervisor 모델이 정해진 형식(`next`)으로 고름 | 모델의 선택은 흔들릴 수 있고, Researcher의 출력 형식을 정하면 흐름이 안정됨 |
| 4. Critic 더하기 (선택) | Supervisor 모델이 Critic 판정을 보고 고름 | 반복을 끝내는 조건이 코드가 아니라 프롬프트에만 있음 |

역할을 나눌 때는 Agent마다 받을 입력과 돌려줄 출력의 형식을 정해야 합니다. 형식이 흐리면 Supervisor처럼 다음 차례를 정하는 쪽도 판단을 잘못할 수 있습니다.

값을 바꿔 다시 실행하고, 결과를 성공 기준(🟢/⛔)과 비교해 스스로 판단했습니다.